# reading BM25 file

In [ ]:
import json

with open(r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\train_claimdecomp_bm25_evidence.json", 'r') as f:
    data = json.load(f)

# Optionally, print the loaded data or its keys to verify
print(data.keys() if isinstance(data, dict) else len(data))

9935


bm25_data is one query’s METADATA+top hits from BM25 retrieval.

- `query_id`: your claim index (0..N-1). a unique id for this input query/claim.
- `doc_id`: list of retrieved document IDs (usually those from your corpus hit by BM25).
- `scores`: same-length list of BM25 relevance scores, one score per returned doc_id.
- `docs`: the text snippets (or full docs) fetched for each doc_id in the same order. 

So row i means: for query_id X, doc_id[i] was retrieved with BM25 score scores[i] and content docs[i].

# taking only top 100 retrieved evidences



In [2]:
# Filter data to keep only top 100 ranked retrievals for each claim
filtered_data = []

for item in data:
    # Create a copy of the item
    filtered_item = item.copy()
    
    # Keep only top 100 documents
    if 'docs' in filtered_item:
        filtered_item['docs'] = filtered_item['docs'][:100]
    if 'doc_id' in filtered_item:
        filtered_item['doc_id'] = filtered_item['doc_id'][:100]
    if 'scores' in filtered_item:
        filtered_item['scores'] = filtered_item['scores'][:100]
    
    filtered_data.append(filtered_item)

# Save to the new file
output_path = r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-top100-BM25.json"
with open(output_path, 'w') as f:
    json.dump(filtered_data, f, indent=2)

print(f"Saved {len(filtered_data)} claims with top 100 retrievals to {output_path}")

Saved 9935 claims with top 100 retrievals to E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-top100-BM25.json


# confirming number of claims

In [3]:
# Check total evidences per claim
evidence_counts = []

for item in filtered_data:
    claim_id = item.get('query_id')
    num_docs = len(item.get('docs', []))
    num_doc_ids = len(item.get('doc_id', []))
    num_scores = len(item.get('scores', []))
    
    evidence_counts.append({
        'query_id': claim_id,
        'num_docs': num_docs,
        'num_doc_ids': num_doc_ids,
        'num_scores': num_scores
    })

# Display summary statistics
print(f"Total claims: {len(evidence_counts)}")
print(f"\nEvidence counts per claim:")
print(f"Min docs: {min(e['num_docs'] for e in evidence_counts)}")
print(f"Max docs: {max(e['num_docs'] for e in evidence_counts)}")
print(f"Avg docs: {sum(e['num_docs'] for e in evidence_counts) / len(evidence_counts):.2f}")

# Check for any inconsistencies
inconsistent = [e for e in evidence_counts if not (e['num_docs'] == e['num_doc_ids'] == e['num_scores'])]
if inconsistent:
    print(f"\nFound {len(inconsistent)} claims with inconsistent evidence counts")
else:
    print("\nAll claims have consistent evidence counts")

Total claims: 9935

Evidence counts per claim:
Min docs: 100
Max docs: 100
Avg docs: 100.00

All claims have consistent evidence counts


<!-- For each claim in "E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\
1-my dataset\1-train-decomposed-claims.json":

1. The JSON has a claim, 5 decomposed sub-questions, and 100 BM25-retrieved 
   evidence passages per claim (in this json file (E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-top100-BM25.json)).

2. For each of the 5 sub-questions, construct the query by concatenating as:
   "Instruct: Retrieve evidence that supports or refutes the claim.\nQuery: 
   Claim: {claim} Question: {sub_question}"

3. Embed all 5 queries using the Linq-Embed-Mistral model loaded locally via 
   sentence-transformers from HuggingFace (Linq-AI-Research/Linq-Embed-Mistral).

4. Embed all 100 evidence passages WITHOUT any instruction prefix. Cache these 
   embeddings since they are the same for all 5 sub-questions of the same claim.

5. Compute cosine similarity between each query embedding and all 100 evidence 
   embeddings.

6. For each sub-question, get the top-1 evidence and its cosine similarity score.

7. Save the output as a JSON file where each entry contains:
   - original claim
   - sub_question
   - top_1_evidence (text)
   - top_1_score (float)


input json structure for claims + decomposed claims looks like this:
{
  "original_id": "...",
  "original_claim": ["...", "...", ...],
  "decomposed_questions": ["...", "...", ...] 
}

and input json structure for 100 evidences looks like this:
{
  "doc_id": "...",
  "scores": ["...", "...", ...],
  "query_id": "" 
  "claim": "" 
  "docs": ["...", "...", ...] 
} -->

In [1]:
from sentence_transformers import SentenceTransformer
import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm




# Load the embedding model
model = SentenceTransformer(r"E:\Embedding Models\Linq-Embed-Mistral-FULL")

# checking the model is running or not 
print("should print \n (1, hidden_dim)")
print(model)
emb = model.encode(["test sentence"])
print(emb.shape)


# Load the decomposed claims
with open(r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-train-decomposed-claims.json", 'r') as f:
    decomposed_claims = json.load(f)

# Load the BM25 evidences (already loaded as filtered_data)
# Create a mapping of query_id to evidence data
evidence_map = {item['query_id']: item for item in filtered_data}

# Output list
output_results = []

# Process each claim
for claim_item in tqdm(decomposed_claims, desc="Processing claims"):
    original_id = claim_item.get('original_id')
    original_claim = claim_item.get('original_claim', [''])[0]  # Take first if it's a list
    decomposed_questions = claim_item.get('decomposed_questions', [])
    
    # Get corresponding evidence
    if original_id not in evidence_map:
        continue
    
    evidence_item = evidence_map[original_id]
    evidence_passages = evidence_item.get('docs', [])
    
    # Embed all evidence passages once (cache them)
    evidence_embeddings = model.encode(evidence_passages, convert_to_numpy=True)
    
    # Process each sub-question
    for sub_question in decomposed_questions:
        # Construct the query with instruction prefix
        query = f"Instruct: Retrieve evidence that supports or refutes the claim.\nQuery: Claim: {original_claim} Question: {sub_question}"
        
        # Embed the query
        query_embedding = model.encode(query, convert_to_numpy=True).reshape(1, -1)
        
        # Compute cosine similarity
        similarities = cosine_similarity(query_embedding, evidence_embeddings)[0]
        
        # Get top-1 evidence
        top_1_idx = np.argmax(similarities)
        top_1_evidence = evidence_passages[top_1_idx]
        top_1_score = float(similarities[top_1_idx])
        
        # Add to results
        output_results.append({
            'original_id': original_id,
            'original_claim': original_claim,
            'sub_question': sub_question,
            'top_1_evidence': top_1_evidence,
            'top_1_score': top_1_score
        })

# Save the output
output_path = r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-reranked.json"
with open(output_path, 'w') as f:
    json.dump(output_results, f, indent=2)

print(f"Saved {len(output_results)} results to {output_path}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

: 

In [8]:
import json
from pathlib import Path

file_path = Path(r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-top100-BM25.json")

with file_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

def word_count(text):
    return len(str(text).split())

max_words = 0
max_evidence = None
total_words = 0
total_evidences = 0
min_words = float('inf')

for item in data:
    evidences = item.get("docs", [])

    for evidence in evidences:
        if isinstance(evidence, str):
            text = evidence
        elif isinstance(evidence, dict):
            text = evidence.get("text", "")
        else:
            text = str(evidence)

        count = word_count(text)

        total_words += count
        total_evidences += 1

        if count > max_words:
            max_words = count
            max_evidence = text
        
        if count < min_words:
            min_words = count
            min_evidence = text

avg_words = total_words / total_evidences if total_evidences > 0 else 0

print("Total evidences:", total_evidences)
print("Maximum words in an evidence:", max_words)
print("Minimum words in an evidence:", min_words)
print("Average words per evidence:", avg_words)
print("\nEvidence with maximum words:\n")
print(min_evidence)
print(max_evidence)


Total evidences: 993500
Maximum words in an evidence: 3248
Minimum words in an evidence: 1
Average words per evidence: 35.81988827377957

Evidence with maximum words:

news/2022/03/24/california-introduces-new-bill-that-would-allow-mothers-to-kill-their-babies-up-to-7-days-after-birth/.
it wasn't that long ago -- just five or six years -- that virtually every major college football team had only one helmet design. the main exception was washington state, which had separate helmets for home and road (which was considered a major eccentricity at the time). things sure have changed. a college football team with only one or two helmets is now viewed as being somewhere between quaint and pathetic. thanks to the explosion in helmet designs -- blackout, whiteout, gray, carbon fiber, camouflage, chrome, pride, stars and stripes, rivalry, matte-finish, throwback and more -- many programs have at least three different helmet colors that they can customize with various decals and face mask colors